<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB/W10/Q2/ASSIGNMENT_W10_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
import zipfile
from pathlib import Path
from dataclasses import dataclass, field


# Database initial state
database = {
    "A": 100,
    "B": 200,
    "C": 300,
    "D": 400
}

# Global timestamp counter
timestamp_counter = 0

# Transaction execution history
transaction_log = []

# Successfully committed transactions
committed_transactions = []

# Failed validation details
failure_details = []


# ==================================================
# TRANSACTION CLASS
# ==================================================

@dataclass
class Transaction:

    transaction_id: str

    read_set: set = field(default_factory=set)

    write_set: dict = field(default_factory=dict)

    start_ts: int = 0

    validation_ts: int = 0

    finish_ts: int = 0

    status: str = "ACTIVE"


# ==================================================
# TIMESTAMP GENERATOR
# ==================================================

def generate_timestamp():

    global timestamp_counter

    timestamp_counter += 1

    return timestamp_counter

Writing validation_protocol.py


In [2]:
# ==================================================
# PHASE 1: READ / EXECUTION
# ==================================================

def read_execution(transaction, key):

    print(
        f"{transaction.transaction_id}: "
        f"Reading {key}"
    )

    if key not in database:

        print("ERROR: Data item does not exist.")

        return None

    value = database[key]

    transaction.read_set.add(key)

    print(
        f"Read successful: {key} = {value}"
    )

    return value


def write_execution(transaction, key, value):

    print(
        f"{transaction.transaction_id}: "
        f"Preparing write {key} = {value}"
    )

    transaction.write_set[key] = value


In [3]:
# ==================================================
# PHASE 2: VALIDATION
# ==================================================

def validate_transaction(transaction):

    print(
        f"\nValidating {transaction.transaction_id}"
    )

    # Assign Validation Timestamp
    transaction.validation_ts = generate_timestamp()

    print(
        f"Start-TS: {transaction.start_ts}"
    )

    print(
        f"Validation-TS: {transaction.validation_ts}"
    )

    # Compare with previously committed transactions
    for previous in committed_transactions:

        # Only transactions validated after this
        # transaction started can conflict.

        if previous.validation_ts <= transaction.start_ts:
            continue

        # Check read-set intersection with previous write-set
        conflict = (
            transaction.read_set &
            set(previous.write_set.keys())
        )

        if conflict:

            transaction.status = "ABORTED"

            transaction.finish_ts = generate_timestamp()

            message = (
                f"Transaction {transaction.transaction_id} "
                f"failed validation.\n"
                f"Conflicting previous transaction: "
                f"{previous.transaction_id}\n"
                f"Current transaction read-set: "
                f"{sorted(transaction.read_set)}\n"
                f"Previous transaction write-set: "
                f"{sorted(previous.write_set.keys())}\n"
                f"Intersecting data items: "
                f"{sorted(conflict)}\n"
                f"Reason: The transaction read data that was "
                f"modified by a transaction that belongs "
                f"to its validation interval.\n"
                f"Decision: ABORTED - Changes not committed.\n"
            )

            failure_details.append(message)

            print("VALIDATION FAILED!")

            print(
                "Conflicting items:",
                conflict
            )

            return False

    print("VALIDATION SUCCESSFUL!")

    return True

In [4]:
# ==================================================
# PHASE 3: WRITE / COMMIT
# ==================================================

def write_phase(transaction):

    print(
        f"\nWrite phase: {transaction.transaction_id}"
    )

    # Commit changes only after successful validation
    for key, value in transaction.write_set.items():

        database[key] = value

        print(
            f"Database updated: {key} = {value}"
        )

    transaction.finish_ts = generate_timestamp()

    transaction.status = "COMMITTED"

    committed_transactions.append(transaction)

    print(
        f"Transaction {transaction.transaction_id} "
        f"COMMITTED."
    )

    print(
        f"Finish-TS: {transaction.finish_ts}"
    )


